# Satellite Overlay Visualization — Northeast India Holdout
**Runtime:** Google Colab — **GPU (T4 or A100)**

**Purpose:** Publication-quality figures with **satellite imagery as the base layer** and
**semi-transparent coloured patch rectangles** overlaid showing predicted/observed connectivity.

The workflow:
1. Load `sampled_patches.csv` + NB17 model predictions
2. Use `contextily` to pull Esri World Imagery satellite basemap tiles on-the-fly
3. Plot satellite base → overlay 6.72 km chip rectangles coloured by value

**Main figures:**
- **(A) Observed** ground truth (`log_mean_tests`) on satellite
- **(B) Predicted** (Prithvi-only / Fusion) on satellite
- **(C) Residuals** (predicted − true) on satellite

**Optional:** Binary overlay (observed vs predicted `has_coverage`).

**Prerequisites:** Run NB17 first for saved XGBoost models.

**Inputs:**
- `data/raw/patches_2019/` — GeoTIFF patches (for Prithvi embedding extraction)
- `data/processed/sampled_patches.csv` — metadata + labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data

## Step 0: Environment Setup

In [ ]:
# Cell 0.1: Clone repo
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

> **Note:** The cell below installs `terratorch` and pins NumPy.  
> After it runs the runtime will **restart automatically**.  
> Re-run from **Step 0.3** after the restart.

In [ ]:
# Cell 0.2: Install dependencies
# TerraTorch MUST be installed first — restart runtime after this cell.

# Pin numpy BEFORE terratorch to prevent 2.x ABI mismatch
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow

# Restart runtime so all C extensions load against the pinned numpy
import os
print("Restarting runtime to apply numpy pin — re-run from next cell after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# Cell 0.3: Clone/cd repo (needed again after runtime restart) + sync patches

import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

# Verify patches are available
patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if patch_count >= 6900:
    print(f'Patches available: {patch_count:,}')
else:
    raise FileNotFoundError(
        f'Only {patch_count} patches found in {PATCH_DIR}. '
        f'Run NB01 first (Steps 1-7) to download patches and save them to Google Drive.'
    )

## Step 1: Imports & Constants

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import torch
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib import cm as mpl_cm
import warnings
import logging
import json
from pathlib import Path
from shapely.geometry import box
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

HLS_MEANS  = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS   = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE      = 0.0001
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']

PATCH_DIR      = 'data/raw/patches_2019'
PATCH_SIZE_M   = 6720.0
PATCH_SIZE_DEG = PATCH_SIZE_M / 111_320.0
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_MODELS   = Path('outputs/models')
OUTPUT_FIGURES  = Path('outputs/figures')
OUTPUT_RESULTS  = Path('outputs/results')
for p in [OUTPUT_FEATURES, OUTPUT_FIGURES, OUTPUT_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels, aggregate
targets (`coverage_fraction`, `log_mean_tests`), stratification features,
and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 4: Extract Features for Test Set
We need engineered features (35-d) + Prithvi embeddings (1024-d) for the test set
to generate predictions from the saved NB17 models.

In [ ]:
# 4A: Engineered features
def extract_engineered_features(patch_path):
    with rasterio.open(patch_path) as src:
        img = src.read().astype(np.float32) * SCALE
    feats = {}
    for i, name in enumerate(BAND_NAMES):
        band  = img[i]
        valid = band[band > 0]
        if len(valid) == 0: valid = np.array([0.0])
        feats[f'{name}_mean'] = float(valid.mean())
        feats[f'{name}_std']  = float(valid.std())
        feats[f'{name}_p10']  = float(np.percentile(valid, 10))
        feats[f'{name}_p90']  = float(np.percentile(valid, 90))
    red, nir, swir1, green = img[2], img[3], img[4], img[1]
    ndvi  = np.where((nir+red)    > 0, (nir-red)    / (nir+red    +1e-8), 0)
    ndbi  = np.where((swir1+nir)  > 0, (swir1-nir)  / (swir1+nir  +1e-8), 0)
    mndwi = np.where((green+swir1)> 0, (green-swir1)/ (green+swir1+1e-8), 0)
    feats['NDVI_mean']       = float(ndvi.mean())
    feats['NDVI_std']        = float(ndvi.std())
    feats['NDBI_mean']       = float(ndbi.mean())
    feats['NDBI_std']        = float(ndbi.std())
    feats['MNDWI_mean']      = float(mndwi.mean())
    feats['NIR_spatial_var'] = float(nir.var())
    feats['bright_frac']     = float((red > 0.15).mean())
    return feats


# We need train features too for median imputation
print('Extracting engineered features for train (for median imputation) ...')
rows_train = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Train'):
    try:
        rows_train.append(extract_engineered_features(f"{PATCH_DIR}/{row['patch_id']}.tif"))
    except Exception:
        rows_train.append({})
X_train_eng = pd.DataFrame(rows_train)
for feat in ['ntl', 'builtup', 'elevation']:
    X_train_eng[feat] = train_df[feat].values
col_med = X_train_eng.median()

print('Extracting engineered features for test ...')
rows_test = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test'):
    try:
        rows_test.append(extract_engineered_features(f"{PATCH_DIR}/{row['patch_id']}.tif"))
    except Exception:
        rows_test.append({})
X_test_eng = pd.DataFrame(rows_test)
for feat in ['ntl', 'builtup', 'elevation']:
    X_test_eng[feat] = test_df[feat].values
X_test_eng = X_test_eng.fillna(col_med)

ENGINEERED_COLS = list(X_test_eng.columns)
X_test_eng_np = X_test_eng.values.astype(np.float32)
print(f'Engineered test features: {X_test_eng_np.shape}')

In [ ]:
# 4B: Load cached Prithvi embeddings OR extract fresh
emb_path = OUTPUT_FEATURES / 'prithvi_frozen_embeds_test.npz'

if emb_path.exists():
    data = np.load(emb_path)
    X_test_prithvi = data['X'].astype(np.float32)
    print(f'Loaded cached Prithvi test embeddings: {X_test_prithvi.shape}')
else:
    print('Cached embeddings not found — extracting from scratch with TerraTorch ...')

    class PrithviPatchDataset(Dataset):
        def __init__(self, df, patch_dir, n_bands=6):
            self.df        = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands   = n_bands
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row  = self.df.iloc[idx]
            path = f"{self.patch_dir}/{row['patch_id']}.tif"
            with rasterio.open(path) as src:
                img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
            img *= SCALE
            for b in range(self.n_bands):
                img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
            img = np.nan_to_num(img, nan=0.0)
            return torch.from_numpy(img).unsqueeze(1), row['patch_id']

    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False
    print(f'Prithvi encoder loaded on {DEVICE}')

    @torch.no_grad()
    def extract_prithvi_embeddings(df, batch_size=64):
        ds = PrithviPatchDataset(df, PATCH_DIR)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
        all_embs = []
        for x, _ in tqdm(loader, desc='Extracting Prithvi embeddings'):
            x = x.to(DEVICE, dtype=torch.float32, non_blocking=True)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))
        return np.concatenate(all_embs, axis=0)

    X_test_prithvi = extract_prithvi_embeddings(test_df)

    # Cache for future runs
    np.savez_compressed(emb_path, X=X_test_prithvi,
                        patch_id=test_df['patch_id'].to_numpy())
    print(f'Embeddings extracted and cached: {X_test_prithvi.shape}')

    del encoder
    torch.cuda.empty_cache()

# Build fusion features
X_test_fusion = np.hstack([X_test_eng_np, X_test_prithvi])
print(f'Prithvi-only test: {X_test_prithvi.shape}')
print(f'Fusion test:       {X_test_fusion.shape}')

## Step 5: Load Saved Models & Generate Predictions

In [ ]:
TARGET = 'log_mean_tests'

# Load NB17 XGBoost models
model_prithvi = xgb.XGBRegressor()
model_prithvi.load_model(str(OUTPUT_MODELS / f'fusion_prithvi_only_{TARGET}.json'))
print(f'Loaded Prithvi-only model: fusion_prithvi_only_{TARGET}.json')

model_fusion = xgb.XGBRegressor()
model_fusion.load_model(str(OUTPUT_MODELS / f'fusion_engineered_prithvi_{TARGET}.json'))
print(f'Loaded Fusion model: fusion_engineered_prithvi_{TARGET}.json')

# Generate predictions
pred_prithvi = model_prithvi.predict(X_test_prithvi)
pred_fusion  = model_fusion.predict(X_test_fusion)

print(f'Prithvi-only predictions — mean: {pred_prithvi.mean():.3f}, std: {pred_prithvi.std():.3f}')
print(f'Fusion predictions       — mean: {pred_fusion.mean():.3f}, std: {pred_fusion.std():.3f}')
print(f'Ground truth             — mean: {test_df[TARGET].mean():.3f}, std: {test_df[TARGET].std():.3f}')

## Step 6: Attach Predictions to Test DataFrame

In [ ]:
test_df['pred_prithvi'] = pred_prithvi
test_df['pred_fusion']  = pred_fusion

print(f'Test set: {len(test_df)} chips in NE India')
print(f'Lon range: [{test_df["lon"].min():.2f}, {test_df["lon"].max():.2f}]')
print(f'Lat range: [{test_df["lat"].min():.2f}, {test_df["lat"].max():.2f}]')
print(f'Ground truth  — mean: {test_df["log_mean_tests"].mean():.3f}')
print(f'Prithvi pred  — mean: {pred_prithvi.mean():.3f}')
print(f'Fusion pred   — mean: {pred_fusion.mean():.3f}')

## Step 7: (Satellite basemap loaded on-the-fly via contextily — no GEE export needed)

## Step 8: Install contextily & Define Overlay Helper
Use `contextily` to pull Esri World Imagery satellite basemap tiles on-the-fly —
no GEE export or local GeoTIFF needed.

In [ ]:
!pip install -q contextily

import contextily as cx
from matplotlib.collections import PatchCollection
from matplotlib.colors import TwoSlopeNorm, Normalize

PAD = 0.5
PLOT_EXTENT = [
    test_df['lon'].min() - PAD, test_df['lon'].max() + PAD,
    test_df['lat'].min() - PAD, test_df['lat'].max() + PAD,
]
VIS_PATCH_DEG = PATCH_SIZE_DEG * 1.3


def make_patch_rect(lon, lat, half=VIS_PATCH_DEG / 2):
    return mpatches.Rectangle((lon - half, lat - half),
                               VIS_PATCH_DEG, VIS_PATCH_DEG)


def plot_overlay(ax, lons, lats, values,
                 cmap='RdYlGn', vmin=None, vmax=None, norm=None,
                 alpha=0.55, title='', cbar_label=''):
    ax.set_xlim(PLOT_EXTENT[0], PLOT_EXTENT[1])
    ax.set_ylim(PLOT_EXTENT[2], PLOT_EXTENT[3])

    cx.add_basemap(ax, crs='EPSG:4326',
                   source=cx.providers.Esri.WorldImagery, zoom=7)

    rects = [make_patch_rect(lon, lat) for lon, lat in zip(lons, lats)]
    if norm is None:
        norm = Normalize(vmin=vmin if vmin is not None else values.min(),
                         vmax=vmax if vmax is not None else values.max())
    cm = plt.get_cmap(cmap)
    colors = [cm(norm(v)) for v in values]

    pc = PatchCollection(rects, facecolors=colors, edgecolors='none',
                         alpha=alpha, transform=ax.transData)
    ax.add_collection(pc)

    sm = plt.cm.ScalarMappable(cmap=cm, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(cbar_label, fontsize=10)

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=12, fontweight='bold')

print('plot_overlay() ready — uses contextily Esri World Imagery basemap.')

## Step 10: Main Overlay Figures — Connectivity Heatmaps
Two 3-panel figures (one per model), each showing the **entire NE India test set**:
- **(A) Observed** (`log_mean_tests` ground truth) — `RdYlGn`, green = high connectivity
- **(B) Predicted** (model output) — same colormap & scale
- **(C) Residuals** (predicted − true) — `RdBu_r` with `TwoSlopeNorm` centred at 0

In [ ]:
TARGET = 'log_mean_tests'
true_vals = test_df[TARGET].values
lons = test_df['lon'].values
lats = test_df['lat'].values

def make_overlay_figure(true_vals, pred_vals, lons, lats,
                        model_label, fignum='1', filename=None):
    residuals = pred_vals - true_vals
    vmax_shared = max(true_vals.max(), pred_vals.max())

    fig, axes = plt.subplots(1, 3, figsize=(22, 8), constrained_layout=True)
    fig.suptitle(
        f'Figure {fignum}: Satellite Imagery with Connectivity Overlay — {TARGET}\n'
        f'Northeast India Test Set (n = {len(true_vals)}) — {model_label}',
        fontsize=14, fontweight='bold')

    # Panel A: Ground truth
    plot_overlay(axes[0], lons, lats, true_vals,
                 cmap='RdYlGn', vmin=0, vmax=vmax_shared,
                 alpha=0.55, title='(A) Observed (Ookla Ground Truth)',
                 cbar_label=TARGET)

    # Panel B: Model prediction
    plot_overlay(axes[1], lons, lats, pred_vals,
                 cmap='RdYlGn', vmin=0, vmax=vmax_shared,
                 alpha=0.55, title=f'(B) Predicted ({model_label})',
                 cbar_label=f'Predicted {TARGET}')

    # Panel C: Residuals (diverging)
    res_abs_max = max(np.percentile(np.abs(residuals), 95), 0.1)
    res_norm = TwoSlopeNorm(vmin=-res_abs_max, vcenter=0, vmax=res_abs_max)
    plot_overlay(axes[2], lons, lats, residuals,
                 cmap='RdBu_r', norm=res_norm,
                 alpha=0.55, title='(C) Residuals (Predicted − True)',
                 cbar_label='Residual')

    if filename:
        plt.savefig(filename, dpi=200, bbox_inches='tight')
        print(f'Saved: {filename}')
    plt.show()
    return fig


# ── Figure 1: Fusion model ────────────────────────────────────
fig1 = make_overlay_figure(
    true_vals, pred_fusion, lons, lats,
    model_label='Eng.+Prithvi Fusion', fignum='1',
    filename=str(OUTPUT_FIGURES / 'satellite_overlay_fusion.png'))

# ── Figure 2: Prithvi-only model ─────────────────────────────
fig2 = make_overlay_figure(
    true_vals, pred_prithvi, lons, lats,
    model_label='Prithvi-only', fignum='2',
    filename=str(OUTPUT_FIGURES / 'satellite_overlay_prithvi.png'))

## Step 11: Binary Overlay Version
2-panel figure: **(A) Observed** binary coverage vs **(B) Predicted** binary coverage
(fusion model at val-optimal threshold), both overlaid on satellite.

In [ ]:
# Load val-optimal thresholds from NB17 metrics
def load_threshold(model_key, target):
    path = OUTPUT_RESULTS / f'fusion_{model_key}_{target}_metrics.csv'
    if path.exists():
        return pd.read_csv(path)['opt_threshold'].iloc[0]
    return None

thr_fusion = load_threshold('engineered_prithvi', TARGET)
if thr_fusion is None:
    thr_fusion = float(np.median(pred_fusion))
    print(f'Warning: threshold not found, using median fallback: {thr_fusion:.4f}')
else:
    print(f'Fusion threshold (val-optimal): {thr_fusion:.4f}')

binary_obs  = test_df['has_coverage'].values.astype(float)
binary_pred = (pred_fusion >= thr_fusion).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
fig.suptitle('Figure 3: Binary Connectivity Detection — Satellite Overlay\n'
             f'Northeast India Test Set (n = {len(test_df)})',
             fontsize=14, fontweight='bold')

# Panel A: Observed binary
plot_overlay(axes[0],
             lons, lats, binary_obs,
             cmap='RdYlGn', vmin=0, vmax=1,
             alpha=0.5, title='(A) Observed Binary Coverage',
             cbar_label='Has Coverage')

# Panel B: Predicted binary (fusion, val-optimal threshold)
plot_overlay(axes[1],
             lons, lats, binary_pred,
             cmap='RdYlGn', vmin=0, vmax=1,
             alpha=0.5,
             title=f'(B) Predicted Binary (Fusion, thr={thr_fusion:.3f})',
             cbar_label='Predicted Coverage')

plt.savefig(OUTPUT_FIGURES / 'satellite_overlay_binary.png',
            dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_FIGURES / "satellite_overlay_binary.png"}')

## Step 12: Summary Statistics — Full NE India Test Set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

gt = test_df[TARGET].values

rows = []
for model_name, preds in [('Prithvi-only', pred_prithvi), ('Fusion', pred_fusion)]:
    residuals = preds - gt
    rows.append({
        'Model': model_name,
        'N chips': len(gt),
        'MAE': mean_absolute_error(gt, preds),
        'RMSE': np.sqrt(mean_squared_error(gt, preds)),
        'Spearman rho': spearmanr(gt, preds).statistic,
        'Mean residual': residuals.mean(),
        'Residual std': residuals.std(),
    })

summary = pd.DataFrame(rows)
print('=== NE India Test Set Summary ===')
print(summary.to_string(index=False))

summary.to_csv(OUTPUT_RESULTS / 'nb20_test_summary.csv', index=False)
print(f'\nSaved: {OUTPUT_RESULTS / "nb20_test_summary.csv"}')

## Step 13: Push to GitHub

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

files = ['notebooks/20_qualitative_case_study.ipynb']
for pat in ['satellite_overlay_*', 'nb20_*']:
    files.extend([str(p) for p in OUTPUT_FIGURES.glob(pat)])
    files.extend([str(p) for p in OUTPUT_RESULTS.glob(pat)])

for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping: {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB20: Satellite overlay visualization — GEE RGB base + PatchCollection overlays'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')